# 绝对位置编码和旋转位置的相对编码
直接开撕

In [1]:
# 每日一导
import torch
import math
import torch.nn as nn

In [2]:
class PositionEncoder(nn.Module):
    def __init__(self,d_model,max_len):
        super().__init__()
        pe = torch.zeros(max_len,d_model) #[T,D]
        position =torch.arange(max_len).unsqueeze(1) #[T,1]
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000) / d_model)
        )     
        pe[:, 0::2] = torch.sin(position * div_term)       # [T, D/2] -> [T, D] 的偶数位
        pe[:, 1::2] = torch.cos(position * div_term)       # [T, D/2] -> [T, D] 的奇数位

        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self,x):
        T = x.shape[1]
        return x + self.pe[:, :T, :]


In [7]:
x = torch.randn(2,4,8)
model = PositionEncoder(d_model = 8,max_len=4)
print(model(x)-x)


tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
           9.9995e-01,  9.9999e-04,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
           9.9980e-01,  2.0000e-03,  1.0000e+00],
         [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
           9.9955e-01,  3.0000e-03,  1.0000e+00]],

        [[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9999e-03,
           9.9995e-01,  9.9999e-04,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
           9.9980e-01,  2.0000e-03,  1.0000e+00],
         [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9996e-02,
           9.9955e-01,  3.0000e-03,  1.0000e+00]

In [2]:
# 现在来撕 RoPE
# 话不多说直接开撕
# 每日一导
import torch
import torch.nn as nn
import math


In [3]:
class RotaryEmbedding(nn.Module):
    def __init__(self,d_model,max_len):
        super().__init__()
        
        freq = 1 / (10000**(torch.arange(0,d_model,2).float()) / d_model)
        pos = torch.arange(max_len)
        freqs = torch.outer(pos,freq)

        self.register_buffer("cos", torch.cos(freqs))
        self.register_buffer("sin", torch.sin(freqs))

    def forward(self,x):
        """
        x = [B,heads,T,heads_dim]
        """
        B,H,T,D = x.shape
        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        x_rot_1 = x1 * cos - x2 * sin
        x_rot_2 = x1 * sin + x2 * cos

        x_out = torch.stack((x_rot_1, x_rot_2), dim=-1)

        return x_out.flatten(-2)


In [7]:
x = torch.ones(2,2,16,8)
model = RotaryEmbedding(d_model=8,max_len=16)
torch.set_printoptions(precision=8, sci_mode=False)
print(model(x)[0,0,:5,:])   # 只看前5个token，别全打印

tensor([[ 1.00000000,  1.00000000,  1.00000000,  1.00000000,  1.00000000,
          1.00000000,  1.00000000,  1.00000000],
        [-1.13485825,  0.84385824,  0.99999994,  1.00000012,  1.00000000,
          1.00000000,  1.00000000,  1.00000000],
        [-0.66975617, -1.24556279,  0.99999982,  1.00000012,  1.00000000,
          1.00000000,  1.00000000,  1.00000000],
        [ 1.32975745, -0.48139936,  0.99999976,  1.00000024,  1.00000000,
          1.00000000,  1.00000000,  1.00000000],
        [ 0.28279662,  1.38565004,  0.99999970,  1.00000036,  1.00000000,
          1.00000000,  1.00000000,  1.00000000]])
